In [ ]:
from pathlib import Path
import sys

BASE_DIR = Path().cwd().parent

PICTURE_DATA_DIR = BASE_DIR / "picture_data"
LEUKEMIA_CANCER_DATA_DIR = PICTURE_DATA_DIR / "cancer_images"
DATASET_DIR = LEUKEMIA_CANCER_DATA_DIR / "cancer_analysis"
TRAIN_DIR = DATASET_DIR / "train"
VALIDATION_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

FINE_TUNING_DIR = BASE_DIR / "leukemia_cancer"
FEATURE_DIR = FINE_TUNING_DIR / "features"
MODEL_DIR = FINE_TUNING_DIR / "models"

CONFUSED_IMAGES_DIR = BASE_DIR / "confused_images" / "cancer"

VERSION = 3
VARIANT = 8

# Variant 1: Basis
# Variant 2: + Klassengewichte
# Variant 3: + Dropout 0.5
# Variant 4: + Data Augmentation
# Variant 5: + Regularization L2 = 0.001
# Variant 6: + BatchNormalization Training
# Variant 7: Learn 0.000005, Drop 0.3
# Variant 8: Densenet121 BaseModel, Learn 0.00001, Drop 0.5, L2 = 0.001, Data Augmentation, Klassengewichte

MODEL_TYPE = "densenet121"

CHECKPOINT_PATH = MODEL_DIR / f"cnn_model_fine_tuned_base_{VARIANT}.keras"

CLASS_NAMES = [
    "Benign",
    "Early",
    "Pre",
    "Pro"
]

sys.path.append(str(BASE_DIR))

from data_handling.manage_data import create_dataset, load_features, get_classweights
from data_handling.create_confusion_images import export_misclassified_images
from model.build_model import save_model, load_model, build_finetuning_model
from evaluation.evaluation import evaluate_model

from model.training_object_classes import EpochInfo, get_early_stopping, get_checkpoint

In [2]:
def fine_tune (class_weights):
    model = build_finetuning_model(
        learning_rate=0.00001, 
        output_count=4, 
        dropout=0.5, 
        regularizer=0.001, 
        batch_trainable=True,
        model_type=MODEL_TYPE
    )

    model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=30,
        shuffle=True,
        class_weight=class_weights,
        callbacks=[
            EpochInfo(), 
            get_early_stopping(patience=4, min_delta=0.001),
            get_checkpoint(CHECKPOINT_PATH)
        ],
        verbose=1
    )

In [3]:
X_train, y_train, X_validation, y_validation, X_test, y_test = load_features(FEATURE_DIR, VERSION)

In [4]:
class_weights = get_classweights(y_train)

In [5]:
fine_tune(class_weights=class_weights)

Epoch 1/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step - accuracy: 0.7036 - loss: 0.8079
Epoche 1:  108.12 Sekunden
System RAM: 70.4 %
System verfügbar: 4.10 GB
Python Prozess: 2.70 GB


Epoch 1: val_loss improved from None to 0.25688, saving model to c:\Users\MarcelKramer\Desktop\Unterrichtsmaterialien\08. Machine Learning\Abschlussprojekt\lung_cancer\models\cnn_model_fine_tuned_base_8.keras

Epoch 1: finished saving model to c:\Users\MarcelKramer\Desktop\Unterrichtsmaterialien\08. Machine Learning\Abschlussprojekt\lung_cancer\models\cnn_model_fine_tuned_base_8.keras
244/244 ━━━━━━━━━━━━━━━━━━━━ 109s 380ms/step - accuracy: 0.7036 - loss: 0.8079 - val_accuracy: 0.9324 - val_loss: 0.2569
Epoch 2/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.9193 - loss: 0.2516
Epoche 2:  92.01 Sekunden
System RAM: 70.8 %
System verfügbar: 4.03 GB
Python Prozess: 2.72 GB


Epoch 2: val_loss improved from 0.25688 to 0.17578, saving model to c:\Users\MarcelKramer\Desktop\Unterrichtsmaterialien

In [6]:
model = load_model(MODEL_DIR, f"fine_tuned_base_{VARIANT}", compile=False)

c:\Users\MarcelKramer\Desktop\Unterrichtsmaterialien\08. Machine Learning\Abschlussprojekt\cnn_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 6 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


In [7]:
evaluation, result_df = evaluate_model(model, X_validation, y_validation, CLASS_NAMES)

evaluation

26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 129ms/step


{'accuracy': 0.972972972972973,
 'balanced_accuracy': 0.9578298981284057,
 'log_loss': 0.07367129623889923,
 'confusion_matrix': array([[106,  19,   0,   1],
        [  0, 246,   0,   0],
        [  0,   0, 241,   0],
        [  1,   0,   1, 199]]),
 'classification_report': {'Benign': {'precision': 0.9906542056074766,
   'recall': 0.8412698412698413,
   'f1-score': 0.9098712446351931,
   'support': 126.0},
  'Early': {'precision': 0.9283018867924528,
   'recall': 1.0,
   'f1-score': 0.9628180039138943,
   'support': 246.0},
  'Pre': {'precision': 0.9958677685950413,
   'recall': 1.0,
   'f1-score': 0.9979296066252588,
   'support': 241.0},
  'Pro': {'precision': 0.995,
   'recall': 0.9900497512437811,
   'f1-score': 0.9925187032418953,
   'support': 201.0},
  'accuracy': 0.972972972972973,
  'macro avg': {'precision': 0.9774559652487428,
   'recall': 0.9578298981284057,
   'f1-score': 0.9657843896040603,
   'support': 814.0},
  'weighted avg': {'precision': 0.9744273050232068,
   'rec

In [8]:
validation_images = create_dataset(VALIDATION_DIR, class_names=CLASS_NAMES, is_preprocess=False).file_paths

Found 814 files belonging to 4 classes.


In [9]:
result_df = result_df.reset_index()

result_df["image_path"] = result_df["index"].apply(lambda x: validation_images[x])

result_df

confusion_filter = result_df["y_true"] != result_df["y_pred"]

confusion_df = result_df[confusion_filter]

confusion_df

,index,y_true,y_pred,y_proba,image_path
0,0,Benign,Early,0.906804,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
17,17,Benign,Early,0.616811,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
29,29,Benign,Early,0.569186,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
38,38,Benign,Early,0.578247,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
47,47,Benign,Early,0.607575,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
53,53,Benign,Early,0.802705,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
59,59,Benign,Early,0.968123,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
62,62,Benign,Early,0.928292,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
68,68,Benign,Early,0.947806,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
72,72,Benign,Early,0.783433,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...


In [10]:
print(confusion_df.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False))

y_true  y_pred
Benign  Early     19
        Pro        1
Pro     Benign     1
        Pre        1
dtype: int64


In [13]:
export_misclassified_images(confusion_df, CONFUSED_IMAGES_DIR, CLASS_NAMES)

In [12]:
test_evaluation, test_result_df = evaluate_model(model, X_test, y_test, CLASS_NAMES)

test_evaluation

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step


{'accuracy': 0.9714285714285714,
 'balanced_accuracy': 0.9539473684210527,
 'log_loss': 0.09747249633073807,
 'confusion_matrix': array([[ 62,  14,   0,   0],
        [  0, 148,   0,   0],
        [  0,   0, 145,   0],
        [  0,   0,   0, 121]]),
 'classification_report': {'Benign': {'precision': 1.0,
   'recall': 0.8157894736842105,
   'f1-score': 0.8985507246376812,
   'support': 76.0},
  'Early': {'precision': 0.9135802469135802,
   'recall': 1.0,
   'f1-score': 0.9548387096774194,
   'support': 148.0},
  'Pre': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 145.0},
  'Pro': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 121.0},
  'accuracy': 0.9714285714285714,
  'macro avg': {'precision': 0.9783950617283951,
   'recall': 0.9539473684210527,
   'f1-score': 0.9633473585787751,
   'support': 490.0},
  'weighted avg': {'precision': 0.9738977072310405,
   'recall': 0.9714285714285714,
   'f1-score': 0.9706244573565751,
   'support': 490.0}}}

In [14]:
test_result_df

,y_true,y_pred,y_proba
0,Benign,Benign,0.994504
1,Benign,Benign,0.999997
2,Benign,Benign,0.868839
3,Benign,Benign,0.999999
4,Benign,Benign,0.967473
...,...,...,...
485,Pro,Pro,0.980171
486,Pro,Pro,0.966217
487,Pro,Pro,0.978022
488,Pro,Pro,0.802523
